# 01 · Backtest against the lines

First look: does the team-points model find value against the market?

- Model: boosting on the ratings residual, `+priors` features (`04_feature_engineering`).
- Out-of-sample: walk-forward predictions for 2019–2023 (each season predicted by a model
  trained only on earlier seasons). 2024 and 2025 stay untouched.
- Spread and total are **derived** from the two team predictions.
- Grading at −110: break-even is **52.4%**.
- **CLV** (closing line value): how many points the line moved toward our pick between
  open and close. Positive CLV means the market later agreed with us. It's the most
  stable early sign of real edge.

⚠️ Many thresholds are checked on the same seasons, so small differences are noise. The
honest number comes from the 2025 test, run once.

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd

from canes_cfb.features import cumulative_sets
from canes_cfb.paths import PROCESSED, RAW
from canes_cfb.validation import walk_forward

features = pd.read_parquet(PROCESSED / "team_games.parquet")
games = pd.read_parquet(RAW / "games.parquet")

## 1. Out-of-sample predictions (walk-forward 2019–2023)

In [ ]:
data = features[
    features.completed
    & features.season.between(2016, 2023)
    & ~features.shortened
    & features.market_points.notna()
].reset_index(drop=True)
cols = cumulative_sets()["+priors"]
y, exp_points = data.points.to_numpy(), data.exp_points.to_numpy()
data["pred"] = np.nan
for _, train, valid in walk_forward(data):
    model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.02,
        num_leaves=7,
        min_child_samples=200,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        verbose=-1,
    ).fit(data.loc[train, cols], (y - exp_points)[train])
    data.loc[valid, "pred"] = exp_points[valid] + model.predict(data.loc[valid, cols])

## 2. Team predictions → game spread and total

In [ ]:
scored = data[data.pred.notna()].merge(
    games[["game_id", "home_id", "home_points", "away_points"]], on="game_id"
)
home = scored[scored.team_id == scored.home_id]
away = scored[scored.team_id != scored.home_id][["game_id", "pred"]]
g = home.merge(away.rename(columns={"pred": "pred_away"}), on="game_id")
g["pred_margin"] = g.pred - g.pred_away
g["pred_total"] = g.pred + g.pred_away
g["margin"] = g.home_points - g.away_points
g["total"] = g.home_points + g.away_points
len(g)

## 3. Win rate by size of disagreement

Spread: the model likes home when `pred_margin + spread > 0`; home covers when
`margin + spread > 0`. Pushes are excluded.

In [ ]:
def by_edge(edge, result, clv=None):
    rows = []
    for threshold in (0, 2, 3, 4, 5, 7):
        pick = (edge.abs() >= threshold) & (result != 0) & edge.notna()
        win = (np.sign(edge[pick]) == result[pick]).mean()
        row = {
            "min edge (pts)": threshold,
            "bets": int(pick.sum()),
            "win %": 100 * win,
            "ROI % at -110": 100 * (win * (1 + 100 / 110) - 1),
        }
        if clv is not None:
            row["avg CLV (pts)"] = clv[pick].mean()
        rows.append(row)
    return pd.DataFrame(rows).round(2)


results = {}
for kind in ("close", "open"):
    edge = g.pred_margin + g[f"spread_{kind}"]
    result = np.sign(g.margin + g[f"spread_{kind}"])
    clv = np.sign(edge) * (g.spread_open - g.spread_close) if kind == "open" else None
    results[f"spread vs {kind}"] = by_edge(edge, result, clv)
    edge_t = g.pred_total - g[f"total_{kind}"]
    result_t = np.sign(g.total - g[f"total_{kind}"])
    clv_t = np.sign(edge_t) * (g.total_close - g.total_open) if kind == "open" else None
    results[f"total vs {kind}"] = by_edge(edge_t, result_t, clv_t)
for name, table in results.items():
    print(name)
    display(table)

## 4. Reading it (2026-09-23)

| | Win % | Verdict |
|---|---|---|
| Spread vs closing line | 47.3–49.1% | No edge. The bigger the disagreement, the worse. |
| Total vs closing line | 47.5–50.5% | No edge. |
| Spread vs opening line | 48.4–50.2% | Below break-even, but **CLV positive and growing with edge** (+0.2 → +1.1 pts). |
| Total vs opening line | 50.7–51.7% up to 5-pt edges (47.7% at 7+) | Close to break-even (52.4%), not above. Small positive CLV. |

**The model isn't profitable yet.** Against the closing line that's expected: it's the
sharpest number in the market. The encouraging part is that the opening line moves
toward the model's picks, and more so the bigger the edge. The model sees information
the market prices in later, but not enough yet to beat the vig.

What would move this (see ROADMAP):
1. Optuna tuning and an ensemble (nothing is tuned yet).
2. Model the market directly: predict the **residual vs. the opening line** or the line
   move, instead of raw points.
3. Better early-season priors (weeks 1–2 remain the weakest).
4. Only bet where the model has shown CLV, and track CLV live in 2026.